# Day 40 — Practice 4 & 5: Hive External Table + Koneksi DBeaver

**Tujuan:** Buat Hive External Table di atas data Parquet yang sudah ada di HDFS, lalu query dari DBeaver.

Alur:
```
HDFS Parquet  →  Hive External Table  →  Query via DBeaver / Spark SQL
(Data Lake)       (Data Warehouse)         (BI / Analytics Tool)
```

**Konsep yang dipraktikkan:**
- Perbedaan **Managed Table** vs **External Table** di Hive
- Buat External Table yang pointnya ke data HDFS
- `MSCK REPAIR TABLE` untuk detect partisi baru
- Koneksi DBeaver ke HiveServer2
- Query Hive dari DBeaver

> **Prasyarat:** Jalankan dulu `02_adventureworks_to_hdfs.ipynb` agar data sudah ada di HDFS.

## Setup SparkSession dengan Hive Support

In [1]:
docker exec adventureworks-postgres psql -U postgres -c "CREATE DATABASE hive_metastore;"
docker exec adventureworks-postgres psql -U postgres -c "\l"
docker restart hive-metastore
docker logs hive-metastore --tail 20
docker restart hiveserver2

SyntaxError: Missing parentheses in call to 'exec'. Did you mean exec(...)? (3195947042.py, line 1)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import subprocess

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Hive-External-Tables') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

print(f'Spark {spark.version} ready dengan Hive support')

# Cek koneksi ke Hive Metastore
spark.sql('SHOW DATABASES').show()

Spark 3.5.0 ready dengan Hive support
+---------+
|namespace|
+---------+
|  default|
+---------+



## 1. Konsep: Managed vs External Table

| | Managed Table | External Table |
|---|---|---|
| **Data lokasi** | Di-manage Hive (`/user/hive/warehouse`) | Di lokasi HDFS yang kita tentukan |
| **DROP TABLE** | Data **ikut terhapus** | Data di HDFS **tetap ada** |
| **Use case** | Data yang sepenuhnya dikelola Hive | Data yang dibuat dari proses lain (Spark, Flume, dll) |
| **Kita pakai** | — | ✅ (data sudah di HDFS dari notebook sebelumnya) |

## 2. Buat Database Hive

In [3]:
# Buat database adventureworks di Hive
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks COMMENT "AdventureWorks Data Warehouse"')
spark.sql('USE adventureworks')

print('=== Databases yang tersedia ===')
spark.sql('SHOW DATABASES').show()

=== Databases yang tersedia ===
+--------------+
|     namespace|
+--------------+
|adventureworks|
|       default|
+--------------+



## 3. Buat External Tables — Tabel Dimensi

Tabel dimensi tidak berpartisi. Langsung point ke folder Parquet.

In [4]:
# Helper: buat external table dari Parquet di HDFS
def create_external_table(table_name: str, hdfs_path: str):
    """
    Buat Hive External Table yang pointnya ke Parquet di HDFS.
    Schema otomatis di-infer dari file Parquet.
    """
    # Baca schema dari Parquet
    df = spark.read.parquet(hdfs_path)
    
    # Register sebagai temp view dulu
    df.createOrReplaceTempView(f'_tmp_{table_name}')
    
    # Buat external table
    spark.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    
    count = spark.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'✓ {table_name:35s} → {count:6,} rows  ({hdfs_path})')

# Buat external table untuk semua tabel dimensi
dim_tables = [
    ('dim_product',           f'{HDFS_BASE}/Production/Product'),
    ('dim_product_category',  f'{HDFS_BASE}/Production/ProductCategory'),
    ('dim_product_subcat',    f'{HDFS_BASE}/Production/ProductSubcategory'),
    ('dim_person',            f'{HDFS_BASE}/Person/Person'),
    ('dim_employee',          f'{HDFS_BASE}/HumanResources/Employee'),
    ('dim_department',        f'{HDFS_BASE}/HumanResources/Department'),
    ('dim_customer',          f'{HDFS_BASE}/Sales/Customer'),
    ('dim_territory',         f'{HDFS_BASE}/Sales/SalesTerritory'),
]

for table_name, path in dim_tables:
    create_external_table(table_name, path)

print('\nSemua tabel dimensi berhasil dibuat di Hive!')

✓ dim_product                         →    504 rows  (hdfs://namenode:9000/datalake/raw/Production/Product)
✓ dim_product_category                →      4 rows  (hdfs://namenode:9000/datalake/raw/Production/ProductCategory)
✓ dim_product_subcat                  →     37 rows  (hdfs://namenode:9000/datalake/raw/Production/ProductSubcategory)
✓ dim_person                          → 19,972 rows  (hdfs://namenode:9000/datalake/raw/Person/Person)
✓ dim_employee                        →    290 rows  (hdfs://namenode:9000/datalake/raw/HumanResources/Employee)
✓ dim_department                      →     16 rows  (hdfs://namenode:9000/datalake/raw/HumanResources/Department)
✓ dim_customer                        → 19,820 rows  (hdfs://namenode:9000/datalake/raw/Sales/Customer)
✓ dim_territory                       →     10 rows  (hdfs://namenode:9000/datalake/raw/Sales/SalesTerritory)

Semua tabel dimensi berhasil dibuat di Hive!


In [5]:
spark.sql("USE adventureworks")
spark.sql("SHOW TABLES").show()

+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|adventureworks|        dim_customer|      false|
|adventureworks|      dim_department|      false|
|adventureworks|        dim_employee|      false|
|adventureworks|          dim_person|      false|
|adventureworks|         dim_product|      false|
|adventureworks|dim_product_category|      false|
|adventureworks|  dim_product_subcat|      false|
|adventureworks|       dim_territory|      false|
|              |   _tmp_dim_customer|       true|
|              | _tmp_dim_department|       true|
|              |   _tmp_dim_employee|       true|
|              |     _tmp_dim_person|       true|
|              |    _tmp_dim_product|       true|
|              |_tmp_dim_product_...|       true|
|              |_tmp_dim_product_...|       true|
|              |  _tmp_dim_territory|       true|
+--------------+--------------------+-----------+


## 4. Buat External Tables — Tabel Fakta (dengan Partisi)

In [7]:
orders_path = f"{HDFS_BASE}/Sales/SalesOrderHeader"

# 1. Baca data langsung dari HDFS
df = spark.read.parquet(orders_path)

# 2. Simpan sebagai tabel eksternal/managed dengan partisi
df.write.format("parquet").mode("overwrite").partitionBy(
    "order_year", "order_month"
).saveAsTable("adventureworks.fact_sales_orders")

print("Tabel berhasil dibuat dan dipartisi!")

Tabel berhasil dibuat dan dipartisi!


In [8]:
# MSCK REPAIR TABLE: scan HDFS dan auto-register semua partisi ke metastore
print('Mendeteksi partisi di HDFS...')
spark.sql('MSCK REPAIR TABLE adventureworks.fact_sales_orders')

print('\n=== Partisi yang terdeteksi ===')
spark.sql('SHOW PARTITIONS adventureworks.fact_sales_orders').show(30)

count = spark.sql('SELECT COUNT(*) FROM adventureworks.fact_sales_orders').collect()[0][0]
print(f'Total rows: {count:,}')

Mendeteksi partisi di HDFS...

=== Partisi yang terdeteksi ===
+--------------------+
|           partition|
+--------------------+
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
+--------------------+
only showing top 30 rows

Total rows: 31,465


In [9]:
# Cek schema dan sample data di path tersebut
df_check = spark.read.parquet(orders_path)
df_check.printSchema()
df_check.show(3)
print(f'Columns: {df_check.columns}')

root
 |-- SalesOrderID: integer (nullable = true)
 |-- RevisionNumber: short (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- Status: short (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- SalesOrderNumber: string (nullable = true)
 |-- PurchaseOrderNumber: string (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- BillToAddressID: integer (nullable = true)
 |-- ShipToAddressID: integer (nullable = true)
 |-- ShipMethodID: integer (nullable = true)
 |-- CreditCardID: integer (nullable = true)
 |-- CreditCardApprovalCode: string (nullable = true)
 |-- CurrencyRateID: integer (nullable = true)
 |-- SubTotal: decimal(19,4) (nullable = true)
 |-- TaxAmt: decimal(19,4) (nullable = true)
 |-- Freight: decimal(19,4) (nullab

In [21]:
spark.sql('DROP TABLE IF EXISTS adventureworks.fact_sales_orders')

spark.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_sales_orders (
        SalesOrderID INT,
        RevisionNumber SMALLINT,
        OrderDate TIMESTAMP,
        DueDate TIMESTAMP,
        ShipDate TIMESTAMP,
        Status SMALLINT,
        OnlineOrderFlag BOOLEAN,
        SalesOrderNumber STRING,
        PurchaseOrderNumber STRING,
        AccountNumber STRING,
        CustomerID INT,
        SalesPersonID INT,
        TerritoryID INT,
        BillToAddressID INT,
        ShipToAddressID INT,
        ShipMethodID INT,
        CreditCardID INT,
        CreditCardApprovalCode STRING,
        CurrencyRateID INT,
        SubTotal DECIMAL(19,4),
        TaxAmt DECIMAL(19,4),
        Freight DECIMAL(19,4),
        TotalDue DECIMAL(19,4),
        Comment STRING,
        rowguid STRING,
        ModifiedDate TIMESTAMP
    )
    PARTITIONED BY (order_year INT, order_month INT)
    STORED AS PARQUET
    LOCATION '{orders_path}'
""")

print('Mendeteksi partisi di HDFS...')
spark.sql('MSCK REPAIR TABLE adventureworks.fact_sales_orders')

spark.sql('SHOW PARTITIONS adventureworks.fact_sales_orders').show(30)

count = spark.sql('SELECT COUNT(*) FROM adventureworks.fact_sales_orders').collect()[0][0]
print(f'Total rows: {count:,}')

Mendeteksi partisi di HDFS...
+--------------------+
|           partition|
+--------------------+
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2011/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2012/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
|order_year=2013/o...|
+--------------------+
only showing top 30 rows

Total rows: 31,465


In [12]:
# Tabel fakta salesorderdetail — partisi order_year
detail_path = f'{HDFS_BASE}/Sales/SalesOrderDetail'

spark.sql('DROP TABLE IF EXISTS adventureworks.fact_order_details')

# 1. Baca data langsung dari HDFS menggunakan Spark Read API
df_detail = spark.read.parquet(detail_path)

# 2. Simpan sebagai tabel eksternal terpartisi
df_detail.write.format("parquet") \
    .mode("overwrite") \
    .partitionBy("order_year") \
    .saveAsTable("adventureworks.fact_order_details")

count = spark.sql('SELECT COUNT(*) FROM adventureworks.fact_order_details').collect()[0][0]
print(f'✓ fact_order_details: {count:,} rows')

✓ fact_order_details: 121,317 rows


## 5. Verifikasi: Query Hive External Tables

In [13]:
print('=== Semua tabel di database adventureworks ===')
spark.sql('SHOW TABLES IN adventureworks').show(30)

=== Semua tabel di database adventureworks ===
+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|adventureworks|        dim_customer|      false|
|adventureworks|      dim_department|      false|
|adventureworks|        dim_employee|      false|
|adventureworks|          dim_person|      false|
|adventureworks|         dim_product|      false|
|adventureworks|dim_product_category|      false|
|adventureworks|  dim_product_subcat|      false|
|adventureworks|       dim_territory|      false|
|adventureworks|  fact_order_details|      false|
|adventureworks|   fact_sales_orders|      false|
|              |   _tmp_dim_customer|       true|
|              | _tmp_dim_department|       true|
|              |   _tmp_dim_employee|       true|
|              |     _tmp_dim_person|       true|
|              |    _tmp_dim_product|       true|
|              |_tmp_dim_product_...|       true|
|  

In [14]:
# Query: Total revenue per tahun via Hive
spark.sql("""
    SELECT
        order_year,
        COUNT(*)                    AS total_orders,
        ROUND(SUM(subtotal), 2)     AS total_subtotal,
        ROUND(SUM(totaldue), 2)     AS total_revenue
    FROM adventureworks.fact_sales_orders
    GROUP BY order_year
    ORDER BY order_year
""").show()

+----------+------------+--------------+-------------+
|order_year|total_orders|total_subtotal|total_revenue|
+----------+------------+--------------+-------------+
|      2011|        1607|   12641672.21|         NULL|
|      2012|        3915|   33524301.33|         NULL|
|      2013|       14182|   43622479.05|         NULL|
|      2014|       11761|   20057928.81|         NULL|
+----------+------------+--------------+-------------+



In [15]:
# Query join: Top produk terlaris dari Hive
spark.sql("""
    SELECT
        p.name                          AS product_name,
        p.color,
        SUM(od.orderqty)                AS total_qty,
        ROUND(SUM(od.linetotal), 2)     AS total_revenue
    FROM adventureworks.fact_order_details od
    JOIN adventureworks.dim_product p ON od.productid = p.productid
    GROUP BY p.name, p.color
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

+---------------------------+------------+---------+-------------+
|product_name               |color       |total_qty|total_revenue|
+---------------------------+------------+---------+-------------+
|ML Crankset                |Black       |68       |NULL         |
|Women's Tights, L          |Black       |2123     |NULL         |
|Half-Finger Gloves, M      |Black       |3464     |NULL         |
|Touring-3000 Blue, 58      |Blue        |482      |NULL         |
|LL Touring Frame - Blue, 44|Blue        |25       |NULL         |
|LL Mountain Front Wheel    |Black       |617      |NULL         |
|Touring-3000 Yellow, 44    |Yellow      |851      |NULL         |
|Road-250 Black, 52         |Black       |1245     |NULL         |
|LL Road Frame - Black, 52  |Black       |1581     |NULL         |
|LL Mountain Pedal          |Silver/Black|449      |NULL         |
+---------------------------+------------+---------+-------------+



## 6. Koneksi DBeaver ke HiveServer2

Setelah tabel berhasil dibuat di Hive, kamu bisa query langsung dari DBeaver.

### Langkah-langkah:

1. **Buka DBeaver** → New Connection
2. **Pilih** `Apache Hive`
3. **Isi connection settings:**

| Setting | Value |
|---------|-------|
| Host | `localhost` |
| Port | `10000` |
| Database | `adventureworks` |
| Username | *(kosongkan)* |
| Password | *(kosongkan)* |

4. **Test Connection** → jika sukses, klik **Finish**
5. Buka SQL Editor, coba query:

```sql
SHOW TABLES;

SELECT * FROM dim_product LIMIT 10;

SELECT order_year, COUNT(*) AS total_orders
FROM fact_sales_orders
GROUP BY order_year;
```

> **Catatan:** HiveServer2 berjalan di container pada port `10000`. Pastikan container `hiveserver2` sudah berjalan dengan `docker-compose ps`.

In [18]:
# Verifikasi HiveServer2 berjalan
result = subprocess.run('docker ps | grep hiveserver2', shell=True, capture_output=True, text=True)
if result.stdout:
    print('✅ HiveServer2 berjalan:')
    print(result.stdout)
    print('\nBuka DBeaver dan connect ke:')
    print('  Host: localhost')
    print('  Port: 10000')
    print('  DB  : adventureworks')
else:
    print('❌ HiveServer2 tidak berjalan. Jalankan: docker-compose up -d hiveserver2')

✅ HiveServer2 berjalan:
4853b9624ab6   apache/hive:3.1.3                      "sh -c /entrypoint.sh"   52 minutes ago   Up 12 minutes             0.0.0.0:10000->10000/tcp, 9083/tcp, 0.0.0.0:10002->10002/tcp   hiveserver2


Buka DBeaver dan connect ke:
  Host: localhost
  Port: 10000
  DB  : adventureworks


## 7. Bonus: Cek Detail Tabel di Hive

In [17]:
# Describe detail tabel — lihat bahwa ini EXTERNAL table
spark.sql('DESCRIBE EXTENDED adventureworks.fact_sales_orders').show(40, truncate=False)

+----------------------------+----------------------------+-------+
|col_name                    |data_type                   |comment|
+----------------------------+----------------------------+-------+
|SalesOrderID                |int                         |NULL   |
|RevisionNumber              |smallint                    |NULL   |
|OrderDate                   |timestamp                   |NULL   |
|DueDate                     |timestamp                   |NULL   |
|ShipDate                    |timestamp                   |NULL   |
|Status                      |smallint                    |NULL   |
|OnlineOrderFlag             |boolean                     |NULL   |
|SalesOrderNumber            |string                      |NULL   |
|PurchaseOrderNumber         |string                      |NULL   |
|AccountNumber               |string                      |NULL   |
|CustomerID                  |int                         |NULL   |
|SalesPersonID               |int               

In [22]:
# Buktikan: DROP external table tidak menghapus data di HDFS
# (jalankan ini jika mau uji coba)

spark.sql('DROP TABLE adventureworks.fact_sales_orders')

DataFrame[]

In [30]:
# Cek HDFS — data masih ada:
result = subprocess.run('docker exec hadoop-namenode hdfs dfs -ls /datalake/raw/Sales/SalesOrderHeader', shell=True, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Found 5 items
-rw-r--r--   3 jovyan supergroup          0 2026-10-05 16:40 /datalake/raw/Sales/SalesOrderHeader/_SUCCESS
drwxr-xr-x   - jovyan supergroup          0 2026-10-05 16:40 /datalake/raw/Sales/SalesOrderHeader/order_year=2011
drwxr-xr-x   - jovyan supergroup          0 2026-10-05 16:40 /datalake/raw/Sales/SalesOrderHeader/order_year=2012
drwxr-xr-x   - jovyan supergroup          0 2026-10-05 16:40 /datalake/raw/Sales/SalesOrderHeader/order_year=2013
drwxr-xr-x   - jovyan supergroup          0 2026-10-05 16:40 /datalake/raw/Sales/SalesOrderHeader/order_year=2014




## Summary

Yang sudah dipraktikkan:
- ✅ Buat database `adventureworks` di Hive
- ✅ Buat **External Table** (bukan Managed) yang pointnya ke Parquet di HDFS
- ✅ `MSCK REPAIR TABLE` untuk auto-detect partisi dari HDFS
- ✅ Query tabel Hive via Spark SQL
- ✅ Cara koneksi DBeaver ke HiveServer2 (port 10000)

**Arsitektur yang sudah berdiri:**
```
PostgreSQL (Source)  →  HDFS Parquet (Data Lake)  →  Hive (Data Warehouse)
                         /datalake/raw/...             adventureworks.fact_*
                                                        adventureworks.dim_*
```

**Next →** Day 41 — Gunakan Spark untuk transformasi data di Hive.